In [1]:
import torchvision.transforms as transforms
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torch.optim import SGD, lr_scheduler
from utils.utils import *
import numpy as np
import matplotlib.pyplot as plt
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('GPU') if str(device) == "cuda:0" else print('GPU not Detected - CPU Selected')
print(f"GPUs Count: {torch.cuda.device_count()}") # Command to show how many GPUs are available

GPU
GPUs Count: 1


In [2]:
my_train_batch_size = 64

In [4]:
from dataloaders.cifar10 import cifar10_dataloaders
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10, CIFAR100
from dataloaders.tinyimagenet import tinyimagenet_dataloaders

def mnist_dataloaders(train_batch_size=my_train_batch_size):
    transform = transforms.Compose([transforms.Lambda(lambda img: img.convert('RGB')), transforms.ToTensor()])
    train_dataset = Subset(MNIST('./data', train=True, transform=transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True)
    return train_loader, test_loader

def fashion_mnist_dataloaders(train_batch_size=my_train_batch_size):
    train_transform = transforms.Compose([transforms.RandomHorizontalFlip(),transforms.Lambda(lambda img: img.convert('RGB')), transforms.ToTensor()])
    test_transform = transforms.Compose([transforms.Lambda(lambda img: img.convert('RGB')), transforms.ToTensor()]) 
    train_dataset = Subset(FashionMNIST('./data', train=True, transform=train_transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = FashionMNIST(root='./data', train=False, download=True, transform=test_transform)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True) 
    return train_loader, test_loader

def cifar10_dataloaders(train_batch_size=my_train_batch_size, data_dir = 'datasets/cifar10'):
    train_transform = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor()])
    test_transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = Subset(CIFAR10(data_dir, train=True, transform=train_transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = CIFAR10(data_dir, train=False, transform=test_transform, download=True)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True)
    return train_loader, test_loader

def cifar100_dataloaders(train_batch_size=128, data_dir = 'datasets/cifar100'):
    train_transform = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor()])
    test_transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = Subset(CIFAR100(data_dir, train=True, transform=train_transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = CIFAR100(data_dir, train=False, transform=test_transform, download=True)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True)
    return train_loader, test_loader

train_loader_1, test_loader_1 = mnist_dataloaders(train_batch_size=my_train_batch_size)
train_loader_2, test_loader_2 = fashion_mnist_dataloaders(train_batch_size=my_train_batch_size)
train_loader_3, test_loader_3 = cifar10_dataloaders(train_batch_size=my_train_batch_size)
train_loader_4, test_loader_4 = cifar100_dataloaders(train_batch_size=my_train_batch_size)
train_loader_5, test_loader_5 = tinyimagenet_dataloaders(train_batch_size=my_train_batch_size)

### All Data_Loaders other than TinyImageNet, have been **twofolded**

In [4]:
len(train_loader_1), len(train_loader_2), len(train_loader_3), len(train_loader_4), len(train_loader_5)

(1562, 1562, 1562, 1562, 1562)

In [5]:
len(test_loader_1), len(test_loader_2), len(test_loader_3), len(test_loader_4), len(test_loader_5)

(100, 100, 100, 100, 100)

### **UNIFIED Model**

In [6]:
my_classes = [10, 10, 10, 100, 200]

class MultiBN2d(nn.Module):
    def __init__(self, num_features):
        super(MultiBN2d, self).__init__()
        self.BN_1 = nn.BatchNorm2d(num_features)
        self.BN_2 = nn.BatchNorm2d(num_features)
        self.BN_3 = nn.BatchNorm2d(num_features)
        self.BN_4 = nn.BatchNorm2d(num_features)
        self.BN_5 = nn.BatchNorm2d(num_features)

    def forward(self, _input, dataset_idx):
        if dataset_idx == 1:
            _output = self.BN_1(_input)
        elif dataset_idx == 2:
            _output = self.BN_2(_input)
        elif dataset_idx == 3:
            _output = self.BN_3(_input)
        elif dataset_idx == 4:
            _output = self.BN_4(_input)
        elif dataset_idx == 5:
            _output = self.BN_5(_input)
        return _output

# Fire Module with Batch Normalization
class Fire(nn.Module):
    def __init__(self, in_channels, squeeze_channels, expand1x1_channels, expand3x3_channels):
        super(Fire, self).__init__()

        # Squeeze: Conv -> BN -> ReLU
        self.squeeze_conv = nn.Conv2d(in_channels, squeeze_channels, kernel_size=1, bias=False)
        self.squeeze_sbn = MultiBN2d(squeeze_channels)
        self.squeeze_relu = nn.ReLU(inplace=True)

        # Expand 1x1: Conv -> BN -> ReLU
        self.expand1x1_conv = nn.Conv2d(squeeze_channels, expand1x1_channels, kernel_size=1, bias=False)
        self.expand1x1_sbn = MultiBN2d(expand1x1_channels)
        self.expand1x1_relu = nn.ReLU(inplace=True)

        # Expand 3x3: Conv -> BN -> ReLU
        self.expand3x3_conv = nn.Conv2d(squeeze_channels, expand3x3_channels, kernel_size=3, padding=1, bias=False)
        self.expand3x3_sbn = MultiBN2d(expand3x3_channels)
        self.expand3x3_relu = nn.ReLU(inplace=True)

    def forward(self, x, dataset_idx):
        x = self.squeeze_relu(self.squeeze_sbn(self.squeeze_conv(x), dataset_idx))
        out1 = self.expand1x1_relu(self.expand1x1_sbn(self.expand1x1_conv(x), dataset_idx))
        out3 = self.expand3x3_relu(self.expand3x3_sbn(self.expand3x3_conv(x), dataset_idx))
        return torch.cat([out1, out3], dim=1)

# SqueezeNet 1.1 with Batch Normalization
class SqueezeNet1_1_SBN_SOL(nn.Module):
    def __init__(self, classes = my_classes):
        super(SqueezeNet1_1_SBN_SOL, self).__init__()
        input_channels = 3
        self.classes = classes

        # Feature extractor:    # Initial Conv -> BN -> ReLU
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=3, stride=1, bias=False)    # stride=2
        self.sbn = MultiBN2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

        # Fire modules
        self.fire1 = Fire(64, 16, 64, 64)      # Fire 2
        self.fire2 = Fire(128, 16, 64, 64)     # Fire 3
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

        self.fire3 = Fire(128, 32, 128, 128)   # Fire 4
        self.fire4 = Fire(256, 32, 128, 128)   # Fire 5
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

        self.fire5 = Fire(256, 48, 192, 192)   # Fire 6
        self.fire6 = Fire(384, 48, 192, 192)   # Fire 7
        self.fire7 = Fire(384, 64, 256, 256)   # Fire 8
        self.fire8 = Fire(512, 64, 256, 256)   # Fire 9
        
        ####### Separate Classification Heads
        self.classifier_heads = nn.ModuleDict({str(data_id + 1): nn.Conv2d(512, c, kernel_size=1) for data_id, c in enumerate(classes)})
        self.classifier_avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Weight initialization
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

        # Initialize all classification heads separately
        for head in self.classifier_heads.values():
            nn.init.normal_(head.weight, mean=0.0, std=0.01)
            if head.bias is not None:
                nn.init.zeros_(head.bias)

    def forward(self, x, dataset_idx):
        x = self.pool1(self.relu(self.sbn(self.conv1(x), dataset_idx)))
        x = self.pool2(self.fire2(self.fire1(x, dataset_idx), dataset_idx))
        x = self.pool3(self.fire4(self.fire3(x, dataset_idx), dataset_idx))
        x = self.fire8(self.fire7(self.fire6(self.fire5(x, dataset_idx), dataset_idx), dataset_idx), dataset_idx)
        x = self.classifier_avgpool(self.classifier_heads[str(dataset_idx)](x))
        return torch.flatten(x, 1)

In [7]:
# Simulate argparse values
class Args:
    batch_size, dim = my_train_batch_size, my_train_batch_size
    epochs = 200
    lr = 0.1
    momentum = 0.9
    wd = 5e-4
args = Args()

############### MODEL SELECTION:
model = SqueezeNet1_1_SBN_SOL().to(device)

##### optimizer:
optimizer = SGD(model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.wd)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, args.epochs)

In [8]:
model.classifier_heads

ModuleDict(
  (1): Conv2d(512, 10, kernel_size=(1, 1), stride=(1, 1))
  (2): Conv2d(512, 10, kernel_size=(1, 1), stride=(1, 1))
  (3): Conv2d(512, 10, kernel_size=(1, 1), stride=(1, 1))
  (4): Conv2d(512, 100, kernel_size=(1, 1), stride=(1, 1))
  (5): Conv2d(512, 200, kernel_size=(1, 1), stride=(1, 1))
)

In [9]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

torch.save(model.state_dict(), 'test_model.pth')

Total parameters: 918282


### **Training Loop**

In [10]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
################################## TRAINING:
best_TA = 0

# Dictionaries to store curves data
data_plots = {"train_total_loss": [], "train_loss1": [], "train_loss2": [], "train_loss3": [], "train_loss4": [], "train_loss5": [], "train_acc1": [], "train_acc2": [], "train_acc3": [], "train_acc4": [], "train_acc5": [], "val_total_loss": [], "val_acc1": [], "val_acc2": [], "val_acc3": [], "val_acc4": [], "val_acc5": [], "val_loss1": [], "val_loss2": [], "val_loss3": [], "val_loss4": [], "val_loss5": []}

for epoch in range(1, args.epochs):
    model.train()
    requires_grad_(model, True)
    accs_1, accs_2, accs_3, accs_4, accs_5, losses_1, losses_2, losses_3, losses_4, losses_5, total_losses = AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()
    i = 0
    for (imgs_1, labels_1), (imgs_2, labels_2), (imgs_3, labels_3), (imgs_4, labels_4), (imgs_5, labels_5) in zip(train_loader_1, train_loader_2, train_loader_3, train_loader_4, train_loader_5):
        i+=1
        imgs_1, labels_1, imgs_2, labels_2, imgs_3, labels_3, imgs_4, labels_4, imgs_5, labels_5 = imgs_1.to(device), labels_1.to(device), imgs_2.to(device), labels_2.to(device), imgs_3.to(device), labels_3.to(device), imgs_4.to(device), labels_4.to(device), imgs_5.to(device), labels_5.to(device)
        optimizer.zero_grad()
        
        logits_1 = model(imgs_1, 1)
        loss_1 = loss_fn(logits_1, labels_1)
        
        logits_2 = model(imgs_2, 2)
        loss_2 = loss_fn(logits_2, labels_2)
        
        logits_3 = model(imgs_3, 3)
        loss_3 = loss_fn(logits_3, labels_3)
        
        logits_4 = model(imgs_4, 4)
        loss_4 = loss_fn(logits_4, labels_4)
        
        logits_5 = model(imgs_5, 5)
        loss_5 = loss_fn(logits_5, labels_5)
        
        ############################################### TOTAL Loss:    
        total_loss = loss_1 + loss_2 + loss_3 + loss_4 + loss_5
        total_loss.backward()
        optimizer.step()

        ##### metrics:
        accs_1.append((logits_1.argmax(1) == labels_1).float().mean().item() * 100)
        accs_2.append((logits_2.argmax(1) == labels_2).float().mean().item() * 100)
        accs_3.append((logits_3.argmax(1) == labels_3).float().mean().item() * 100)
        accs_4.append((logits_4.argmax(1) == labels_4).float().mean().item() * 100)
        accs_5.append((logits_5.argmax(1) == labels_5).float().mean().item() * 100)
        losses_1.append(loss_1.item())
        losses_2.append(loss_2.item())
        losses_3.append(loss_3.item())
        losses_4.append(loss_4.item())
        losses_5.append(loss_5.item())
        total_losses.append(total_loss.item())
        if i % 500 == 0:
            train_str = 'Epoch %d-%d | Loss: %.4f,  Acc_1: %.2f,  Acc_2: %.2f,  Acc_3: %.2f,  Acc_4: %.2f,  Acc_5: %.2f' % (epoch, i, total_losses.avg, accs_1.avg, accs_2.avg, accs_3.avg, accs_4.avg, accs_5.avg)
            print(train_str)
   
    scheduler.step()    # lr scheduler update at the end of each epoch:

    ################################## Validation:
    model.eval()
    requires_grad_(model, False)

    eval_this_epoch = (epoch == 1) or (epoch % 5 == 0) or (epoch>=int(0.75*args.epochs)) # boolean
    if eval_this_epoch:
        val_accs_1, val_accs_2, val_accs_3, val_accs_4, val_accs_5, val_losses_1, val_losses_2, val_losses_3, val_losses_4, val_losses_5, val_total_losses = AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()
            
        for i, (imgs, labels) in enumerate(test_loader_1):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 1)    # idx is the num of classes in SOL
            val_loss_1 = loss_fn(logits, labels)
            val_losses_1.append(val_loss_1.item())
            val_accs_1.append((logits.argmax(1) == labels).float().mean().item()*100)

        for i, (imgs, labels) in enumerate(test_loader_2):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 2)    # idx is the num of classes in SOL
            val_loss_2 = loss_fn(logits, labels)
            val_losses_2.append(val_loss_2.item())
            val_accs_2.append((logits.argmax(1) == labels).float().mean().item()*100)
        
        for i, (imgs, labels) in enumerate(test_loader_3):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 3)    # idx is the num of classes in SOL
            val_loss_3 = loss_fn(logits, labels)
            val_losses_3.append(val_loss_3.item())
            val_accs_3.append((logits.argmax(1) == labels).float().mean().item()*100)
        
        for i, (imgs, labels) in enumerate(test_loader_4):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 4)    # idx is the num of classes in SOL
            val_loss_4 = loss_fn(logits, labels)
            val_losses_4.append(val_loss_4.item())
            val_accs_4.append((logits.argmax(1) == labels).float().mean().item()*100)
        
        for i, (imgs, labels) in enumerate(test_loader_5):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 5)    # idx is the num of classes in SOL
            val_loss_5 = loss_fn(logits, labels)
            val_losses_5.append(val_loss_5.item())
            val_accs_5.append((logits.argmax(1) == labels).float().mean().item()*100)
              
        ####### Record Data for Curves
        data_plots["train_acc1"].append(accs_1.avg)
        data_plots["train_acc2"].append(accs_2.avg)
        data_plots["train_acc3"].append(accs_3.avg)
        data_plots["train_acc4"].append(accs_4.avg)
        data_plots["train_acc5"].append(accs_5.avg)
        data_plots["train_loss1"].append(losses_1.avg)
        data_plots["train_loss2"].append(losses_2.avg)
        data_plots["train_loss3"].append(losses_3.avg)
        data_plots["train_loss4"].append(losses_4.avg)
        data_plots["train_loss5"].append(losses_5.avg)
        data_plots["train_total_loss"].append(total_losses.avg)
        
        val_total_losses = val_losses_1.avg + val_losses_2.avg + val_losses_3.avg + val_losses_4.avg + val_losses_5.avg
        data_plots["val_acc1"].append(val_accs_1.avg)
        data_plots["val_acc2"].append(val_accs_2.avg)
        data_plots["val_acc3"].append(val_accs_3.avg)
        data_plots["val_acc4"].append(val_accs_4.avg)
        data_plots["val_acc5"].append(val_accs_5.avg)
        data_plots["val_loss1"].append(val_losses_1.avg)
        data_plots["val_loss2"].append(val_losses_2.avg)
        data_plots["val_loss3"].append(val_losses_3.avg)
        data_plots["val_loss4"].append(val_losses_4.avg)
        data_plots["val_loss5"].append(val_losses_5.avg)
        data_plots["val_total_loss"].append(val_total_losses)
        
    if eval_this_epoch:
        val_str = '\n*********** Epoch %d | Test Acc_1: %.2f | Test Acc_2: %.2f | Test Acc_3: %.2f | Test Acc_4: %.2f | Test Acc_5: %.2f\n' % (epoch, val_accs_1.avg, val_accs_2.avg, val_accs_3.avg, val_accs_4.avg, val_accs_5.avg)
        print(val_str)

    # Save MODEL:
    if eval_this_epoch:
        avg_val_TA_list = []
        avg_val_TA_list.append(val_accs_1.avg)
        avg_val_TA_list.append(val_accs_2.avg)
        avg_val_TA_list.append(val_accs_3.avg)
        avg_val_TA_list.append(val_accs_4.avg)
        avg_val_TA_list.append(val_accs_5.avg)
        avg_val_TA = sum(avg_val_TA_list) / len(avg_val_TA_list)    # Averaging performance across all lambdas
        
        if avg_val_TA >= best_TA:
            best_TA = avg_val_TA # update best TA
            torch.save(model.state_dict(), 'UNIFIED_SqueezeNet1_1_5_Datasets.pth')

Epoch 1-500 | Loss: 11.5533,  Acc_1: 91.12,  Acc_2: 76.20,  Acc_3: 35.22,  Acc_4: 7.24,  Acc_5: 3.27
Epoch 1-1000 | Loss: 10.6231,  Acc_1: 94.24,  Acc_2: 80.57,  Acc_3: 43.29,  Acc_4: 10.98,  Acc_5: 5.56
Epoch 1-1500 | Loss: 10.0288,  Acc_1: 95.41,  Acc_2: 82.55,  Acc_3: 48.20,  Acc_4: 14.13,  Acc_5: 7.49

*********** Epoch 1 | Test Acc_1: 97.83 | Test Acc_2: 85.88 | Test Acc_3: 55.27 | Test Acc_4: 20.31 | Test Acc_5: 8.80

Epoch 2-500 | Loss: 8.2693,  Acc_1: 97.91,  Acc_2: 87.47,  Acc_3: 62.54,  Acc_4: 24.48,  Acc_5: 13.79
Epoch 2-1000 | Loss: 8.1104,  Acc_1: 97.97,  Acc_2: 87.83,  Acc_3: 63.67,  Acc_4: 25.74,  Acc_5: 14.68
Epoch 2-1500 | Loss: 7.9713,  Acc_1: 98.07,  Acc_2: 87.96,  Acc_3: 64.83,  Acc_4: 26.79,  Acc_5: 15.54
Epoch 3-500 | Loss: 7.4632,  Acc_1: 98.14,  Acc_2: 88.24,  Acc_3: 68.68,  Acc_4: 31.13,  Acc_5: 18.98
Epoch 3-1000 | Loss: 7.3956,  Acc_1: 98.10,  Acc_2: 88.27,  Acc_3: 69.04,  Acc_4: 31.95,  Acc_5: 19.47
Epoch 3-1500 | Loss: 7.3562,  Acc_1: 98.06,  Acc_2: 88.41, 

### **Inference**

In [ ]:
checkpoint = torch.load('UNIFIED_SqueezeNet1_1_5_Datasets.pth')
model.load_state_dict(checkpoint)

In [15]:
def test_model(model, test_loader, dataset_idx):
    model.eval()
    requires_grad_(model, False)
    test_accs = {}
    test_accs = AverageMeter()

    for i, (imgs, labels) in enumerate(test_loader):
        imgs, labels = imgs.cuda(), labels.cuda()
        logits = model(imgs, dataset_idx)
        test_accs.append((logits.argmax(1) == labels).float().mean().item()*100)
    print(f"Test Accuracy: {test_accs.avg:.2f} %")

In [17]:
test_model(model, test_loader_1, 1)
test_model(model, test_loader_2, 2)
test_model(model, test_loader_3, 3)
test_model(model, test_loader_4, 4)
test_model(model, test_loader_5, 5)

Test Accuracy: 99.41 %
Test Accuracy: 93.12 %
Test Accuracy: 91.06 %
Test Accuracy: 69.18 %
Test Accuracy: 56.76 %
